<a href="https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula06a_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

class SLPClassifier(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200, learning_rate=0.01):
    self.max_iter = max_iter
    self.learning_rate = learning_rate
    self.model = None

  def fit(self, X, y):
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1]
    output_shape = len(self.labels)

    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(ids, dtype=torch.long)

    self.model = nn.Sequential(
        nn.Linear(input_shape, output_shape),
        nn.Softmax(dim=1)
    )
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    for epoch in range(self.max_iter):
      optimizer.zero_grad()
      outputs = self.model(X)
      loss = criterion(outputs, y)
      loss.backward()
      optimizer.step()

    return self

  def predict_proba(self, X):
    X = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
      outputs = self.model(X)
    return outputs.numpy()

  def predict(self, X):
    proba = self.predict_proba(X)
    idx_pred = np.argmax(proba, axis=1)
    return self.labels[idx_pred]

model = SLPClassifier(10000, 0.0001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9611111111111111


In [21]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

class SHLPClassifier(BaseEstimator, ClassifierMixin):
  def __init__(self, n_hidden=100, max_iter=200, learning_rate=0.01):
    self.n_hidden = n_hidden
    self.max_iter = max_iter
    self.learning_rate = learning_rate
    self.model = None

  def fit(self, X, y):
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1]
    output_shape = len(self.labels)

    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(ids, dtype=torch.long)

    self.model = nn.Sequential(
        nn.Linear(input_shape, self.n_hidden),
        nn.ReLU(),
        nn.Linear(self.n_hidden, output_shape),
        nn.Softmax(dim=1)
    )
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    for epoch in range(self.max_iter):
      optimizer.zero_grad()
      outputs = self.model(X)
      loss = criterion(outputs, y)
      loss.backward()
      optimizer.step()

    return self

  def predict_proba(self, X):
    X = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
      outputs = self.model(X)
    return outputs.numpy()

  def predict(self, X):
    proba = self.predict_proba(X)
    idx_pred = np.argmax(proba, axis=1)
    return self.labels[idx_pred]

model = SHLPClassifier(100, 10000, 0.0001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9861111111111112


In [24]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

class MLPClassfier(BaseEstimator, ClassifierMixin):
  def __init__(self, n_hidden=[100], max_iter=200, learning_rate=0.01):
    self.n_hidden = n_hidden
    self.max_iter = max_iter
    self.learning_rate = learning_rate
    self.model = None

  def fit(self, X, y):
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1]
    output_shape = len(self.labels)

    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(ids, dtype=torch.long)

    layers = [nn.Linear(input_shape, self.n_hidden[0]), nn.ReLU()]
    for i in range(1, len(self.n_hidden)):
      layers.append(nn.Linear(self.n_hidden[i-1], self.n_hidden[i]))
      layers.append(nn.ReLU())
    layers.append(nn.Linear(self.n_hidden[-1], output_shape))
    layers.append(nn.Softmax(dim=1))
    self.model = nn.Sequential(*layers)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    for epoch in range(self.max_iter):
      optimizer.zero_grad()
      outputs = self.model(X)
      loss = criterion(outputs, y)
      loss.backward()
      optimizer.step()

    return self

  def predict_proba(self, X):
    X = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
      outputs = self.model(X)
    return outputs.numpy()

  def predict(self, X):
    proba = self.predict_proba(X)
    idx_pred = np.argmax(proba, axis=1)
    return self.labels[idx_pred]

model = MLPClassfier([100], 10000, 0.0001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9833333333333333


In [25]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total number of parameters in model.model: {total_params}")

Total number of parameters in model.model: 7510


In [31]:
n_feats = X_train.shape[1]
n_ihw = n_feats * 100
n_how = 100 * len(set(y_train))
print(f"Number of input hidden weights: {n_ihw + n_how}")

Number of input hidden weights: 7400


In [32]:
model = MLPClassfier([50, 50], 10000, 0.0001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9777777777777777


In [33]:
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total number of parameters in model.model: {total_params}")

Total number of parameters in model.model: 6310


In [34]:
n_feats = X_train.shape[1]
n_ihw = n_feats * 50
n_hhw = 50 * 50
n_how = 50 * len(set(y_train))
print(f"Number of input hidden weights: {n_ihw + n_hhw + n_how}")

Number of input hidden weights: 6200
